In [0]:
# COMMAND ----------
# CELL 1: Setup & Parameters
import re
import json
import urllib.request
import urllib.error
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, TimestampType, DoubleType, BooleanType
)
from sodapy import Socrata

dbutils.widgets.text("socrata_domain", "opendata.maryland.gov", "Socrata Domain")
dbutils.widgets.text("dataset_id", "", "Single Dataset ID")
dbutils.widgets.text("catalog_name", "open_data_portal", "Catalog Name")
dbutils.widgets.text("schema_name", "bronze", "Target Schema Name")  # <-- All tables land in bronze
dbutils.widgets.text("meta_schema", "meta", "Control Schema Name")
dbutils.widgets.dropdown("verbose", "true", ["true", "false"], "Verbose Logging")

SOCRATA_DOMAIN = dbutils.widgets.get("socrata_domain")
DATASET_ID = dbutils.widgets.get("dataset_id").strip()
CATALOG_NAME = dbutils.widgets.get("catalog_name")
SCHEMA_NAME = dbutils.widgets.get("schema_name")
META_SCHEMA = dbutils.widgets.get("meta_schema")
VERBOSE = dbutils.widgets.get("verbose").lower() == "true"

QUEUE_TABLE = f"`{CATALOG_NAME}`.`{META_SCHEMA}`.`_socrata_ingestion_queue`"

if not DATASET_ID:
    raise ValueError("dataset_id widget must be provided for single-task ingestion.")

def log(*args, **kwargs):
    if VERBOSE:
        print(*args, **kwargs)

def sanitize_table_name(name: str) -> str:
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', (name or '').lower())
    clean = re.sub(r'_+', '_', clean).strip('_')
    return clean[:128]

def sql_str(val) -> str:
    if val is None:
        return ""
    return str(val).strip().replace("\\", "\\\\").replace("'", "\\'")

def run_sql(label: str, stmt: str) -> bool:
    try:
        spark.sql(stmt)
        log(f"    [SUCCESS] {label}")
        return True
    except Exception as err:
        log(f"    [WARNING] {label} failed: {str(err).splitlines()[0]}")
        return False

def append_log_event(dataset_id: str, title: str, category: str, portal_update: str, status: str, records: int = 0, error: str = ""):
    """
    Appends an immutable audit row to the queue table instead of updating in place.
    Preserves audit history and prevents write lock contention.
    """
    schema = StructType([
        StructField("dataset_id", StringType(), False),
        StructField("title", StringType(), True),
        StructField("category", StringType(), True),
        StructField("last_portal_update", StringType(), True),
        StructField("status", StringType(), True),
        StructField("records_ingested", LongType(), True),
        StructField("error_message", StringType(), True)
    ])

    log_row = [{
        "dataset_id": dataset_id,
        "title": title,
        "category": category,
        "last_portal_update": portal_update,
        "status": status,
        "records_ingested": records,
        "error_message": str(error)[:1000] if error else None
    }]

    (spark.createDataFrame(log_row, schema=schema)
        .withColumn("logged_at", current_timestamp())
        .select(
            "dataset_id", "title", "category", "last_portal_update",
            "status", "records_ingested", "error_message", "logged_at"
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(QUEUE_TABLE))

In [0]:
# COMMAND ----------
# CELL 2: Ingest, Transform, and Save Single Unified Table
SECRET_SCOPE = "socrata_scope"
try:
    APP_TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key="app_token")
    log("Loaded Socrata app token from secret scope.")
except Exception:
    APP_TOKEN = None
    log("Warning: Running unauthenticated.")

client = Socrata(SOCRATA_DOMAIN, APP_TOKEN, timeout=600)

GEO_TYPES = {
    'point', 'multipoint', 'linestring', 'multilinestring', 
    'polygon', 'multipolygon', 'location', 'geometry'
}

log(f"\nProcessing Dataset: {DATASET_ID}")

# 1. Fetch Upstream Metadata & Validate Public Status
try:
    dataset_metadata = client.get_metadata(DATASET_ID)
except urllib.error.HTTPError as http_err:
    if http_err.code in (403, 404):
        append_log_event(DATASET_ID, DATASET_ID, "Uncategorized", None, "DEPRECATED", 0, f"HTTP {http_err.code}")
        dbutils.notebook.exit(f"DEPRECATED_HTTP_{http_err.code}")
    raise http_err
except Exception as err:
    append_log_event(DATASET_ID, DATASET_ID, "Uncategorized", None, "FAILED", 0, str(err))
    raise RuntimeError(f"Metadata fetch failed: {err}")

dataset_title = dataset_metadata.get('name', DATASET_ID)
category = dataset_metadata.get("category") or "Uncategorized"
portal_updated_at = (dataset_metadata.get("rowsUpdatedAt") or dataset_metadata.get("dataUpdatedAt"))

approvals = (dataset_metadata.get("approvals") or [{}])[0] or {}
target_audience = (approvals.get("targetAudience") or "").lower()
publication_stage = (dataset_metadata.get("publicationStage") or "").lower()

if not (publication_stage == "published" and target_audience == "public"):
    append_log_event(DATASET_ID, dataset_title, category, portal_updated_at, "DEPRECATED", 0, "Not public/published")
    dbutils.notebook.exit("Filtered non-public dataset.")

table_name = sanitize_table_name(dataset_title)
target_table_path = f"`{CATALOG_NAME}`.`{SCHEMA_NAME}`.`{table_name}`"

# 2. Schema Discovery from Socrata Column Definitions
socrata_columns = dataset_metadata.get('columns', [])
all_meta_columns = [c.get('fieldName') for c in socrata_columns if c.get('fieldName')]

geo_cols = set()
socrata_type_cast_map = {}

for c in socrata_columns:
    fn = c.get('fieldName')
    dt = (c.get('dataTypeName') or '').lower()
    if not fn:
        continue
    if dt in GEO_TYPES:
        geo_cols.add(fn)
    elif dt in ['number', 'money']:
        socrata_type_cast_map[fn] = 'double'
    elif dt == 'checkbox':
        socrata_type_cast_map[fn] = 'boolean'

serverless_input_schema = StructType([
    StructField(col_name, StringType(), True) for col_name in all_meta_columns
])
cast_exprs = {c: col(c).cast(t) for c, t in socrata_type_cast_map.items()}

# 3. Download SODA Rows
offset = 0
BATCH_SIZE = 50000 if len(all_meta_columns) <= 25 else 20000
all_records = []

try:
    while True:
        try:
            records = client.get(DATASET_ID, limit=BATCH_SIZE, offset=offset, order=":id")
        except Exception:
            records = client.get(DATASET_ID, limit=BATCH_SIZE, offset=offset)

        if not records:
            break

        for r in records:
            row_dict = {}
            for col_name in all_meta_columns:
                val = r.get(col_name)
                if val is None:
                    row_dict[col_name] = None
                elif isinstance(val, (dict, list)):
                    row_dict[col_name] = json.dumps(val)
                else:
                    val_str = str(val).strip()
                    if col_name in socrata_type_cast_map and socrata_type_cast_map[col_name] == 'double':
                        val_str = re.sub(r'[\$,]', '', val_str)
                    row_dict[col_name] = val_str if val_str else None
            all_records.append(row_dict)

        offset += BATCH_SIZE
        if len(records) < BATCH_SIZE:
            break

    total_records = len(all_records)
    if total_records == 0:
        append_log_event(DATASET_ID, dataset_title, category, portal_updated_at, "EMPTY", 0, None)
        dbutils.notebook.exit("EMPTY")

    # 4. In-Memory Spark DataFrame & Spatial Detection
    df = spark.createDataFrame(all_records, schema=serverless_input_schema)
    if cast_exprs:
        df = df.withColumns(cast_exprs)

    loc_col = next((c for c in geo_cols if c in df.columns), None)
    if not loc_col:
        lower_cols = {c.lower(): c for c in df.columns}
        loc_col = next((lower_cols[c] for c in ["location", "the_geom", "geom", "geometry"] if c in lower_cols), None)

    sample_val = ""
    if loc_col:
        sample_row = df.select(loc_col).filter(col(loc_col).isNotNull() & (F.trim(col(loc_col)) != "")).limit(1).collect()
        sample_val = str(sample_row[0][loc_col]).strip() if sample_row else ""

    has_geometry = False

    # Check for Socrata JSON (nested human_address or coordinates)
    if loc_col and ("human_address" in sample_val or "latitude" in sample_val):
        # Extract address elements as standard strings; zip is cast to clean string
        df = (
            df.withColumn("street_address", F.col(loc_col).cast("string"))  # fallback placeholder
              .withColumn("street_address", F.get_json_object(F.get_json_object(col(loc_col), "$.human_address"), "$.address"))
              .withColumn("city", F.get_json_object(F.get_json_object(col(loc_col), "$.human_address"), "$.city"))
              .withColumn("state", F.get_json_object(F.get_json_object(col(loc_col), "$.human_address"), "$.state"))
              .withColumn("zip", F.expr(f"cast(get_json_object(get_json_object({loc_col}, '$.human_address'), '$.zip') AS STRING)"))
              .withColumn("__lat", F.expr(f"try_cast(get_json_object({loc_col}, '$.latitude') AS DOUBLE)"))
              .withColumn("__lon", F.expr(f"try_cast(get_json_object({loc_col}, '$.longitude') AS DOUBLE)"))
        )
        # Check if coordinates actually exist in the payload
        has_coords = df.filter(col("__lat").isNotNull() & col("__lon").isNotNull()).limit(1).count() > 0
        if has_coords:
            has_geometry = True
            df = df.withColumn(
                "__g",
                F.expr("""
                    CASE 
                        WHEN __lon BETWEEN -180 AND 180 AND __lat BETWEEN -90 AND 90 AND NOT (__lon = 0 AND __lat = 0)
                        THEN st_point(__lon, __lat, 4326)
                        ELSE NULL 
                    END
                """)
            )

    # Check for separate numeric lat/lon columns
    elif "latitude" in [c.lower() for c in df.columns] and "longitude" in [c.lower() for c in df.columns]:
        l_map = {c.lower(): c for c in df.columns}
        lat_c, lon_c = l_map["latitude"], l_map["longitude"]
        df = (
            df.withColumn("__lat", F.expr(f"try_cast({lat_c} AS DOUBLE)"))
              .withColumn("__lon", F.expr(f"try_cast({lon_c} AS DOUBLE)"))
        )
        has_coords = df.filter(col("__lat").isNotNull() & col("__lon").isNotNull()).limit(1).count() > 0
        if has_coords:
            has_geometry = True
            df = df.withColumn(
                "__g",
                F.expr("""
                    CASE 
                        WHEN __lon BETWEEN -180 AND 180 AND __lat BETWEEN -90 AND 90 AND NOT (__lon = 0 AND __lat = 0)
                        THEN st_point(__lon, __lat, 4326)
                        ELSE NULL 
                    END
                """)
            )

    # Check for GeoJSON string
    elif loc_col and '"coordinates"' in sample_val:
        has_geometry = True
        df = df.withColumn(
            "__g",
            F.expr(f"CASE WHEN {loc_col} IS NOT NULL AND length(trim({loc_col})) > 0 THEN st_setsrid(st_geomfromgeojson({loc_col}), 4326) ELSE NULL END")
        )

    # Ensure any existing 'zip' or 'zip_code' column is explicitly cast to String
    for z_candidate in ["zip", "zip_code", "zipcode", "postal_code"]:
        if z_candidate in df.columns:
            df = df.withColumn(z_candidate, col(z_candidate).cast("string"))

    # Only append geom, geom_md, and h3_r8 if geometry was detected
    if has_geometry:
        df = (
            df.withColumn("geom", F.expr("__g"))
              .withColumn("geom_md", F.expr("st_transform(__g, 6487)"))
              .withColumn(
                  "h3_r8",
                  F.expr("""
                      CASE 
                          WHEN geom IS NULL OR st_isempty(geom) THEN NULL
                          WHEN abs(st_x(st_centroid(geom))) > 180 OR abs(st_y(st_centroid(geom))) > 90 THEN NULL
                          ELSE h3_longlatash3(st_x(st_centroid(geom)), st_y(st_centroid(geom)), 8)
                      END
                  """)
              )
        )

    # Clean intermediate scratch columns, raw location column, and Socrata internals
    drop_candidates = ["__g", "__lat", "__lon"]
    if loc_col:
        drop_candidates.append(loc_col)

    cols_to_drop = [
        c for c in df.columns 
        if c in drop_candidates 
        or c.startswith("@computed_region") 
        or c.startswith(":")
    ]
    df = df.drop(*cols_to_drop)

    # 5. Single Delta Write
    (
        df.write
          .format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(target_table_path)
    )
    log(f"Committed unified table: {target_table_path} ({total_records} rows).")

    # Set base auto-optimize table properties
    table_props = [
        "'delta.autoOptimize.optimizeWrite' = 'true'",
        "'delta.autoOptimize.autoCompact' = 'true'"
    ]
    if has_geometry:
        table_props.append("'delta.dataSkippingStatsColumns' = 'h3_r8'")

    spark.sql(f"""
        ALTER TABLE {target_table_path} SET TBLPROPERTIES (
            {', '.join(table_props)}
        )
    """)

    # 6. Apply Table Metadata & Comments
    run_sql("Table Comment", f"COMMENT ON TABLE {target_table_path} IS '{sql_str((dataset_metadata.get('description') or ''))}'")

    md = dataset_metadata.get("metadata") or {}
    custom_fields = md.get("custom_fields") or {}
    time_period_custom = custom_fields.get("Time Period") or {}
    agency_custom = custom_fields.get("Agency") or {}
    jurisdiction_custom = custom_fields.get("Jurisdiction") or {}
    data_provided_by = dataset_metadata.get("attribution") or ""

    tbl_properties = {
        "open_data.title": dataset_metadata.get("name"),
        "open_data.url": f"https://{SOCRATA_DOMAIN}/d/{DATASET_ID}",
        "open_data.category": category,
        "open_data.agency": agency_custom.get("State Agency Performing Data Updates", data_provided_by),
        "open_data.jurisdiction": jurisdiction_custom.get("Jurisdiction", "State of Maryland"),
        "open_data.update_frequency": time_period_custom.get("Update Frequency"),
        "open_data.time_period_of_content": time_period_custom.get("Time Period of Content"),
        "socrata.domain": SOCRATA_DOMAIN,
        "socrata.dataset_id": DATASET_ID,
        "source.system": "Socrata Open Data Portal"
    }
    prop_clauses = [f"'{sql_str(k)}' = '{sql_str(v)}'" for k, v in tbl_properties.items() if sql_str(v)]
    if prop_clauses:
        props_sql = ",\n        ".join(prop_clauses)
        run_sql("Properties", f"ALTER TABLE {target_table_path} SET TBLPROPERTIES (\n        {props_sql}\n)")

    raw_tags = dataset_metadata.get("tags") or []
    tag_candidates = [re.sub(r'[^a-zA-Z0-9_]', '_', str(t).strip().lower()).strip('_') for t in list(raw_tags) + [category]]
    unique_tags = list(dict.fromkeys([t for t in tag_candidates if t]))[:MAX_TAGS]

    # Format raw tags and append the key-value tag 'lifecycle' = 'current'
    formatted_tag_clauses = [f"'{sql_str(tag)}'" for tag in unique_tags]
    formatted_tag_clauses.append("'lifecycle' = 'current'")

    if formatted_tag_clauses:
        tags_sql = ", ".join(formatted_tag_clauses)
        run_sql(f"{len(formatted_tag_clauses)} Tags", f"ALTER TABLE {target_table_path} SET TAGS ({tags_sql})")

    # 7. Log Success
    append_log_event(
        dataset_id=DATASET_ID,
        title=dataset_title,
        category=category,
        portal_update=portal_updated_at,
        status="SUCCESS",
        records=total_records,
        error=None
    )

except Exception as pipeline_err:
    append_log_event(DATASET_ID, dataset_title, category, portal_updated_at, "FAILED", 0, str(pipeline_err))
    raise RuntimeError(f"Ingestion failed for {DATASET_ID}: {pipeline_err}")